# Ders 12 - Ajans Not Defteri ile Sohbet Geçmişi Azaltma

Bu not defteri, Microsoft Ajans Çerçevesi kullanarak uzun sohbetlerde bağlamı nasıl yöneteceğinizi gösterir. Sohbetler büyüdükçe, token sayısı artar — sonunda modelin bağlam penceresini aşar. Bunu **bağlam özetleme deseni** ve kalıcı hafıza için bir **ajans not defteri** ile ele alıyoruz.

## Öğrenecekleriniz:
1. **Bağlam Yönetiminin Önemi**: Token sınırlarını ve bağlam pencerelerini anlama
2. **Bağlamdan Haberdar Ajanslar**: Kendi sohbet bağlamlarını yöneten ajanslar oluşturma
3. **Bağlam Özetleme Deseni**: Sohbet geçmişini yoğunlaştırmak için araçlar kullanma
4. **Ajans Not Defteri**: Bağlam azaltımından sonra ayakta kalan kalıcı hafıza

## Önkoşullar:
- Ortam değişkenleri yapılandırılmış Azure OpenAI kurulumu
- Önceki derslerden temel ajans kavramlarının anlaşılması


## Kurulum


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Neden Bağlam Yönetimi Önemlidir

Her LLM'nin sonlu bir **bağlam penceresi** vardır — tek bir istekte işleyebileceği maksimum token sayısı. Çok aşamalı bir konuşma ilerledikçe:

- **Token sayısı kullanıcı mesajı ve asistan yanıtıyla doğrusal olarak artar.**
- **İsteme tokenları maliyetin büyük kısmını oluşturur** çünkü tüm geçmiş her aşamada yeniden gönderilir.
- Sonunda, konuşma **bağlam penceresini aşar** ve model ya kısaltır ya da hata verir.

### Bağlam Yönetimi Stratejileri

| Strateji | Nasıl Çalışır | Takas |
|---|---|---|
| **Kısaltma** | En eski mesajları atar | Erken bağlam kaybolur |
| **Özetleme** | Eski mesajları bir özet halinde yoğunlaştırır | Bazı ayrıntılar kaybolur, ancak temel noktalar korunur |
| **Scratchpad / Harici Bellek** | Anahtar bilgileri konuşmanın dışına depolar | Araç çağrıları gerektirir, ancak herhangi bir azalmadan etkilenmez |

Bu not defterinde, konuşma geçmişi yoğunlaştırılsa bile ajanın sürekliliği koruyabilmesi için **özetleme**yi **scratchpad aracıyla** birleştiriyoruz.


## Bağlam Farkındalığına Sahip Bir Ajan Oluşturma


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Uzun Bir Konuşmayı Simüle Etme

Bağlamın nasıl biriktiğini görmek için çok turlu bir sohbeti inceleyelim. Ajan, tur boyunca önemli ayrıntıları (tercihler, bütçe, seyahat tarihleri) korumalı ve sürekliliği göstermelidir.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Ajanın önceki turlardan bağlamı nasıl koruduğuna dikkat edin — Japonya, suşi, tapınaklar, fotoğrafçılık, 3000$ bütçe, tek başına seyahat ve Nisan zaman dilimi hakkında bilgi sahibi. Kısa bir konuşmada bu iyi çalışır, ancak konuşma uzadıkça tüm geçmişi yeniden göndermek maliyetli hale gelir.

Bağlam birikimini görmek için konuşmaya daha fazla tur ekleyerek devam edelim:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Bağlam Özetleme Deseni

Konuşma büyüdükçe, birikmiş bağlamı sıkıştırılmış bir biçime dönüştürmek için **özetleme aracı** kullanabiliriz. Temsilci, daha eski mesajlar silinse bile temel bilgilerin korunması için önemli tercihleri kaydetmek amacıyla bu aracı çağırır.

Bu desen, daha gelişmiş geçmiş azaltımının yapı taşıdır:
1. Temsilci, konuşmadan önemli gerçekleri belirler
2. Bunları korumak için özetleme aracını çağırır
3. Özet önemli olanı yakaladığı için eski mesajlar güvenle kaldırılabilir

Aşağıda, temsilcinin öğrendiklerinin sıkıştırılmış bir özetini kaydetmek için çağırabileceği `summarize_preferences` aracını tanımlıyoruz.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Özet

Bu derste, Microsoft Agent Framework kullanarak uzun süreli ajan konuşmalarında bağlamı nasıl yöneteceğinizi öğrendiniz:

### Temel Kavramlar
- **Bağlam pencereleri sınırlıdır** — konuşma geçmişindeki her token para maliyeti taşır ve limite sayılır.
- **Özetleme araçları**, ajanların biriktirilmiş bağlamı özlü özetlere dönüştürmesini sağlar, token kullanımını azaltırken temel bilgileri korur.
- **Ajan not defterleri**, herhangi bir konuşma kısaltmasına rağmen kalıcı dışsal belleği sağlar.

### Yaptıklarınız
- Çok turlu konuşmalarda sürekliliği koruyan **bağlam farkındalıklı bir ajan**
- Önemli kullanıcı bilgilerini özlü bir formatta kaydeden bir **özetleme aracı** (`summarize_preferences`)
- Bağlam tutma ve değişim yönetimini gösteren bir **çok turlu konuşma**

### Gerçek Dünya Uygulamaları
- **Müşteri Hizmet Botları**: Uzun destek oturumları boyunca tercihleri hatırlar
- **Kişisel Asistanlar**: Bağlamı tekrar açıklamadan devam eden projeleri takip eder
- **Eğitsel Eğitmenler**: Çoklu etkileşimlerde öğrenci ilerlemesini korur

### Sonraki Adımlar
- Dosya tabanlı kalıcılıkla tam bir not defteri aracı uygulamak
- Özetlemeden sonra otomatik geçmiş kısaltma eklemek
- Anlamsal bellek araması için vektör veritabanlarıyla birleştirmek
- Tam bağlamla günler sonra konuşmaları sürdürebilen ajanlar oluşturmak


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:  
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba göstermemize rağmen, otomatik çevirilerin hata veya yanlışlıklar içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Önemli bilgiler için profesyonel insan çevirisi tavsiye edilir. Bu çevirinin kullanımı sonucunda doğabilecek yanlış anlamalar veya yanlış yorumlamalar nedeniyle sorumluluk kabul edilmemektedir.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
